In [1]:
import random
import numpy as np
from dataclasses import dataclass, field
from typing import List

# --- Configuration Parameters ---
NUM_DUS = 5                  # Scenario A: 5 DUs
TOTAL_USERS = 200            # Total users in system
USERS_PER_DU = TOTAL_USERS // NUM_DUS
REQ_RATE_PER_MIN = 10        # Lambda: 10 requests per minute per DU
SIMULATION_DURATION = 10     # Minutes to simulate

In [4]:
# --- Video Structure Definitions ---
@dataclass
class VideoStructure:
    """Defines the structure of a single video request."""
    video_id: int
    chunks: int = 30
    tiles_per_chunk: int = 12 # 3 rows x 4 cols
    layers: List[str] = field(default_factory=lambda: ['Base', 'High-Quality'])
    
    def total_data_units(self):
        """Calculates total cacheable items in this video."""
        return self.chunks * self.tiles_per_chunk * len(self.layers)

@dataclass
class Request:
    """Represents a generated video request."""
    timestamp: float    # Time of arrival (in minutes)
    du_id: int         # The DU handling the request
    user_id: int       # The specific user making the request
    video_id: int      # The requested video
    details: VideoStructure

class TrafficSimulator:
    def __init__(self, num_dus, users_per_du, arrival_rate):
        self.num_dus = num_dus
        self.users_per_du = users_per_du
        self.arrival_rate = arrival_rate # Lambda
        self.request_log = []

    def generate_requests(self, duration_minutes):
        print(f"--- Starting Simulation for {duration_minutes} minutes ---")
        print(f"Configuration: {self.num_dus} DUs, {self.users_per_du} Users/DU")
        print(f"Traffic Model: Poisson Process ($\lambda$={self.arrival_rate} req/min/DU)")
        print("-" * 50)

        # Simulate each DU independently
        for du_id in range(self.num_dus):
            current_time = 0.0
            
            # Define the user ID range for this DU (Uniform Distribution)
            # e.g., DU 0 has users 0-39, DU 1 has 40-79
            start_user = du_id * self.users_per_du
            end_user = start_user + self.users_per_du - 1
            
            while True:
                # 1. Calculate Inter-arrival time using Exponential Distribution
                # If events are Poisson distributed, time between events is Exponential
                inter_arrival_time = random.expovariate(self.arrival_rate)
                
                current_time += inter_arrival_time
                
                # Stop if we exceed simulation duration
                if current_time > duration_minutes:
                    break
                
                # 2. Select a random User from this DU's pool
                user = random.randint(start_user, end_user)
                
                # 3. Create the Video Request Structure
                # (Assuming a random video ID from a pool of 100)
                vid_id = random.randint(1, 100) 
                vid_struct = VideoStructure(video_id=vid_id)
                
                req = Request(
                    timestamp=current_time,
                    du_id=du_id,
                    user_id=user,
                    video_id=vid_id,
                    details=vid_struct
                )
                
                self.request_log.append(req)

        # Sort all logs by timestamp to see the system-wide timeline
        self.request_log.sort(key=lambda x: x.timestamp)
        return self.request_log

# --- Run the Simulation ---
if __name__ == "__main__":
    # Initialize Simulator
    sim = TrafficSimulator(NUM_DUS, USERS_PER_DU, REQ_RATE_PER_MIN)
    
    # Generate Data
    requests = sim.generate_requests(SIMULATION_DURATION)
    
    # --- Display Statistics ---
    print(f"\nTotal Requests Generated: {len(requests)}")
    print(f"Expected Requests: ~{NUM_DUS * REQ_RATE_PER_MIN * SIMULATION_DURATION}")
    
    print("\nSample Request Stream (First 5 requests):")
    print(f"{'Time (min)':<12} | {'DU ID':<6} | {'User ID':<8} | {'Video Info'}")
    print("-" * 60)
    
    for req in requests[:5]:
        print(f"{req.timestamp:<12.4f} | {req.du_id:<6} | {req.user_id:<8} | Video {req.video_id}")
        
    # Show detailed structure of one request
    print(f"\n[Detail] Structure of Video {requests[0].video_id}:")
    print(f"  Chunks: {requests[0].details.chunks}")
    print(f"  Tiles per Chunk: {requests[0].details.tiles_per_chunk} (3x4 Grid)")
    print(f"  Layers: {requests[0].details.layers}")
    print(f"  Total Cacheable Items: {requests[0].details.total_data_units()}")

--- Starting Simulation for 10 minutes ---
Configuration: 5 DUs, 40 Users/DU
Traffic Model: Poisson Process ($\lambda$=10 req/min/DU)
--------------------------------------------------

Total Requests Generated: 485
Expected Requests: ~500

Sample Request Stream (First 5 requests):
Time (min)   | DU ID  | User ID  | Video Info
------------------------------------------------------------
0.0064       | 3      | 121      | Video 50
0.0163       | 0      | 8        | Video 18
0.0301       | 3      | 155      | Video 75
0.0406       | 1      | 57       | Video 52
0.0445       | 0      | 11       | Video 94

[Detail] Structure of Video 50:
  Chunks: 30
  Tiles per Chunk: 12 (3x4 Grid)
  Layers: ['Base', 'High-Quality']
  Total Cacheable Items: 720


<>:33: SyntaxWarning: invalid escape sequence '\l'
<>:33: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_686664/609554156.py:33: SyntaxWarning: invalid escape sequence '\l'
  print(f"Traffic Model: Poisson Process ($\lambda$={self.arrival_rate} req/min/DU)")
